# Notebook explicatif détaillé : `LSTM_Opti.py`

Ce notebook reprend **tout le fichier `LSTM_Opti.py`** en le découpant en blocs logiques.
Pour chaque bloc de code, tu as **une cellule markdown juste au-dessus** qui explique
en détail (presque ligne par ligne) ce que fait le code qui suit.

Les grandes sections sont :

1. Imports et bibliothèques utilisées  
2. Dataset temporel `SatelliteSequenceDataset`  
3. Modèle LSTM `LSTMModel`  
4. Early stopping `EarlyStopping`  
5. Entraîneur générique `LSTMTrainer`  
6. Gestionnaire d'expériences `ExperimentRunner`  
7. Préparation des données (lecture CSV, features, cibles, DataLoader)  
8. Lancement des expériences et plots de comparaison avec SGP4 / Horizons

Tu peux naviguer cellule par cellule pour te concentrer sur chaque partie.


## 1. Imports et bibliothèques

Ce bloc importe toutes les bibliothèques nécessaires.

- `numpy as np` : calcul numérique, tableaux multidimensionnels, fonctions mathématiques.
- `torch` : coeur de PyTorch (tenseurs, calcul sur GPU/CPU).
- `torch.nn as nn` : modules de réseaux de neurones (LSTM, Linear, fonctions de perte, etc.).
- `Dataset`, `DataLoader` : outils PyTorch pour gérer les données de manière structurée
  et les charger par mini-batchs pendant l'entraînement.
- `matplotlib.pyplot as plt` : pour tracer des courbes (loss, erreurs, etc.).
- `typing` (`Dict`, `Any`, `List`) : annotations de types (lisibilité et aide des IDE).
- `pandas as pd` : manipulation de données tabulaires (le CSV de télémétrie et d'orbite).
- `time` : mesurer précisément la durée de chaque epoch d'entraînement.

Chaque ligne d'import prépare une fonctionnalité que tu retrouveras dans les blocs suivants.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from typing import Dict, Any, List
import pandas as pd
import time  


## 2. Dataset temporel : `SatelliteSequenceDataset`

Cette classe hérite de `torch.utils.data.Dataset`. Elle sert à :

- stocker les séries temporelles d'entrée `X` et les cibles `Y`,
- découper la série en **fenêtres de longueur fixe** `seq_len`,
- séparer les données en **train** et **validation** selon `train_ratio`,
- fournir au `DataLoader` des couples `(séquence, cible)`.

Détail des éléments importants :

- `__init__(self, X, Y, seq_len=128, train_ratio=0.8)` :  
  - `X` : toutes les features par pas de temps (par ex. temps, paramètres orbitaux, position SGP4, etc.).  
  - `Y` : les cibles (par ex. erreur à corriger sur la position).  
  - `seq_len` : nombre de pas de temps consécutifs utilisés pour **une** séquence (c'est la taille d'un batch).  
  - `train_ratio` : fraction de la série utilisée pour l'entraînement (le reste pour la validation).  
  - on découpe `X` et `Y` en `X_train`, `Y_train` d'un côté et `X_valid`, `Y_valid` de l'autre.

- `self.train_mode = True` :  
  - indique si le dataset est en mode **train** ou **validation**.

- `set_mode(self, mode="train")` :  
  - si `mode == "train"`, on servira les données d'entraînement ;  
  - sinon, les données de validation.

- `__len__(self)` :  
  - calcule le nombre de **séquences** possibles en tenant compte de `seq_len`.  
  - on ne peut pas commencer une fenêtre trop près de la fin, sinon on dépasserait.

- `__getitem__(self, idx)` :  
  - construit la séquence `X_seq` : les `seq_len` pas de temps à partir de `idx`.  
  - prend comme cible `y` la valeur à l'instant **final** de cette fenêtre.  
  - retourne `(X_seq, y)` que le `DataLoader` passera ensuite au modèle.

L'idée : « à partir des `seq_len` derniers pas de temps, prédire la correction (ou la cible) au dernier pas de la séquence ».


In [ ]:
class SatelliteSequenceDataset(Dataset):
    def __init__(self, X, Y, seq_len=128, train_ratio=0.8):
        self.X = X
        self.Y = Y
        self.seq_len = seq_len

        assert X.shape[0] == Y.shape[0], "X and Y must align in time"

        split = int(X.shape[0] * train_ratio)
        self.split = split  # 🔹 pour retrouver l’index global validation

        self.X_train, self.Y_train = X[:split], Y[:split]
        self.X_valid, self.Y_valid = X[split:], Y[split:]

        self.train_mode = True

    def set_mode(self, mode="train"):
        self.train_mode = (mode == "train")

    def __len__(self):
        data = self.X_train if self.train_mode else self.X_valid
        return len(data) - self.seq_len

    def __getitem__(self, idx):
        X_data = self.X_train if self.train_mode else self.X_valid
        Y_data = self.Y_train if self.train_mode else self.Y_valid

        X_seq = X_data[idx : idx + self.seq_len]        # (seq_len, features)
        y = Y_data[idx + self.seq_len - 1]              # prédiction du dernier pas

        return torch.tensor(X_seq, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)




## 3. Modèle LSTM : `LSTMModel`

Ce bloc définit le réseau de neurones LSTM utilisé pour apprendre la correction sur la position.

- La classe hérite de `nn.Module` (tous les modèles PyTorch étant stockés dedans).
- Paramètres principaux du constructeur :
  - `input_size` : dimension du vecteur d'entrée à chaque pas de temps (nombre de features).  
  - `hidden_size` : dimension de l'état caché du LSTM (taille de la mémoire interne).  
  - `num_layers` : nombre de couches LSTM empilées.  
  - `output_size` : taille de la sortie (par ex. 3 si on corrige x, y, z).

À l'intérieur :

- `self.lstm = nn.LSTM(...)` :  
  - `input_size` : nombre de features en entrée à chaque pas (dans notre cas 18 et si on enlève "tle_index" on passe à 17).  
  - `hidden_size` : taille du vecteur caché (On essayera 64, 128 et 256 pour tester la dépendance des prédiction à la temporalité : En gros, c’est la “mémoire” interne du LSTM → plus elle est grande :

✔️ le modèle peut apprendre des relations temporelles plus complexes
✔️ il peut modéliser des dynamiques longues ou non linéaires
✔️ il devient plus puissant, plus flexible).  

  - `num_layers` : profondeur du LSTM (On va essayer 1, 2 et 3 si le temps de calcul le permet).  
  - `batch_first=True` : les tenseurs auront la forme `(batch_size, seq_len, input_size)`. C'est une commande historique de pytorch poru faire comprendere au module que l'la première taille est la taille du batch et non la taille de la séquence.

- `self.fc = nn.Linear(hidden_size, output_size)` :  
  - couche linéaire finale qui prend le dernier état caché du LSTM (un vecteur de taille `hidden_size`)  
  - et le projette dans un espace de dimension `output_size` (la prédiction finale).

Dans `forward(self, x)` :

- `lstm_out, _ = self.lstm(x)` :  
  - `lstm_out` contient **tous** les états cachés à chaque pas de temps  
    → forme `(batch_size, seq_len, hidden_size)`.
  - les états finaux `(h_n, c_n)` ne sont pas réutilisés ici (d'où `_`).

- `last_hidden = lstm_out[:, -1, :]` :  
  - on prend le **dernier pas temporel** pour chaque élément du batch.  
  - on obtient un tenseur de taille `(batch_size, hidden_size)`.

- `out = self.fc(last_hidden)` :  
  - on passe ce dernier état caché dans la couche linéaire.  
  - on obtient une prédiction de taille `(batch_size, output_size)`.

- `return out` : renvoie cette prédiction (correction ou cible selon ce que l'on entraîne).


In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.1 if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 3)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.fc(last)

#le forwrard est équivalent à : 

def forward(self, x):
    batch_size = x.size(0)

    # état initial du LSTM : h0 et c0
    h_t = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)
    c_t = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)

    # LSTM interne : va dérouler la séquence automatiquement
    out, (h_t, c_t) = self.lstm(x, (h_t, c_t))

    # h_t contient le dernier état caché
    last_hidden_state = h_t[-1]

    # Prédiction finale
    y = self.fc(last_hidden_state)
    return y



class LSTMMain(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # On regroupe tous les poids en deux grosses matrices :
        # x_t → 4 * hidden_size      (i, f, g, o)
        # h_{t-1} → 4 * hidden_size
        self.W_x = nn.Linear(input_size, 4 * hidden_size, bias=True)
        self.W_h = nn.Linear(hidden_size, 4 * hidden_size, bias=True)

    def forward(self, x_t, h_prev, c_prev):
        """
        x_t    : (batch_size, input_size)
        h_prev : (batch_size, hidden_size)
        c_prev : (batch_size, hidden_size)
        """

        # Combinaison linéaire de x_t et h_prev
        gates = self.W_x(x_t) + self.W_h(h_prev)
        # gates : (batch_size, 4 * hidden_size)

        # On découpe en 4 blocs : i, f, g, o
        i_t, f_t, g_t, o_t = torch.chunk(gates, 4, dim=1)

        # 🔹 Fonctions d'activation explicites :
        i_t = torch.sigmoid(i_t)   # input gate
        f_t = torch.sigmoid(f_t)   # forget gate
        o_t = torch.sigmoid(o_t)   # output gate
        g_t = torch.tanh(g_t)      # candidate cell state

        # 🔹 Nouveau c_t : mélange entre ancien c_prev et nouvelle info g_t
        c_t = f_t * c_prev + i_t * g_t

        # 🔹 Nouveau h_t : sortie "visible" du bloc
        h_t = o_t * torch.tanh(c_t)

        return h_t, c_t



## 4. Early stopping : `EarlyStopping`

Ce bloc implémente une logique d'**early stopping** : arrêter l'entraînement lorsque la
loss de validation ne s'améliore plus pendant un certain nombre d'epochs.

- Le constructeur prend typiquement :
  - `patience` : nombre d'epochs consécutives autorisées sans amélioration.  
  - `min_delta` : amélioration **minimale** sur la loss pour considérer qu'il y a du progrès.

Les attributs internes :
- `self.patience` : valeur de patience enregistrée.
- `self.min_delta` : seuil d'amélioration minimal par rapport à la meilleure valeure de MSE sur l'ensemble de validation atteinte jusqu'ici.
- `self.counter` : compte combien d'epochs d'affilée **sans amélioration** on a déjà eues.
- `self.best_loss` : meilleure loss de validation vue jusqu'ici (initialisée à `np.inf`).
- `self.early_stop` : booléen indiquant s'il faut arrêter ou non.

La méthode `__call__(self, loss)` : 
- On la passe la loss de validation de l'epoch courante.
- Si `loss < best_loss - min_delta` :
  - on a une vraie amélioration → on met à jour `best_loss` et on remet `counter` à 0.
- Sinon :
  - pas d'amélioration suffisante → on incrémente `counter`.
- Si `counter >= patience` :
  - on fixe `early_stop = True` (le reste du code pourra stopper la boucle d'entraînement).


In [ ]:
class EarlyStopping:
    def __init__(self, patience=20, min_delta=1e-5):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, loss):
        if loss < self.best_loss - self.min_delta:
            self.best_loss = loss
            self.counter = 0
        else:
            self.counter += 1
        
        if self.counter >= self.patience:
            self.early_stop = True




## 5. Entraîneur générique : `LSTMTrainer`

Cette classe encapsule tout ce qui concerne **l'entraînement** du modèle LSTM :

- gestion du device (`mps` sur Mac ou `cpu` sinon),
- déplacement du modèle sur le bon device,
- création de l'optimizer Adam,
- définition de la fonction de perte MSE,
- gestion du scheduler de learning rate,
- boucles d'entraînement (`train_epoch`) et de validation (`eval_epoch`),
- boucle globale multi-epochs (`fit`).

Points clés du constructeur :

- `self.device = torch.device(...)` :  
  - teste si `torch.backends.mps.is_available()` est vrai, sinon bascule en `"cpu"`.

- `self.model = model.to(self.device)` :  
  - déplace tous les paramètres du modèle sur le device choisi. Evite de le faire pour tout les paramètres un par un. 

- `self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)` :  
  - optimizer Adam avec le learning rate passé en argument.

- `self.loss_fn = nn.MSELoss()` :  
  - fonction de perte utilisée (erreur quadratique moyenne). On est dans un problème de régression continue en 3D donc la mse est adaptée. 

- `self.history = {...}` :  
  - dictionnaire pour stocker l'historique `train_mse`, `valid_mse`, `train_rmse`, `valid_rmse`, `epoch_time`.

- `self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(...)` :  
  - surveille la métrique de validation (MSE).
  - quand ça stagne trop longtemps, réduit le learning rate d'un facteur (par ex. 0.5).

Dans `train_epoch(self, loader)` :

- met le modèle en mode entraînement : `self.model.train()`.
- boucle sur les batchs `(X, y)` produits par `loader` :
  - déplace `X` et `y` sur le bon device,
  - `optimizer.zero_grad()` pour remettre les gradients à zéro,
  - `y_pred = self.model(X)` pour obtenir les prédictions,
  - `loss = self.loss_fn(y_pred, y)` pour calculer la MSE du batch,
  - `loss.backward()` pour la backpropagation,
  - `optimizer.step()` pour mettre à jour les paramètres,
  - stocke `loss.item()` dans une liste.
- à la fin, calcule la moyenne des pertes (`train_mse`) et sa racine (`train_rmse`).

Dans `eval_epoch(self, loader)` :

- met le modèle en mode évaluation : `self.model.eval()`.
- désactive le calcul de gradient avec `torch.no_grad()` (plus rapide, moins de mémoire).
- boucle similaire, **mais sans backward ni step** :
  - on calcule seulement `y_pred` et la loss, qu'on accumule.
- retourne la MSE et la RMSE de validation.

Dans `fit(self, train_loader, valid_loader, epochs, patience)` :

- crée une instance d'`EarlyStopping` avec la patience voulue.
- pour chaque epoch :
  - mesure le temps de début,
  - appelle `train_epoch` puis `eval_epoch`,
  - calcule la durée de l'epoch,
  - donne `valid_mse` au `scheduler` via `self.scheduler.step(valid_mse)`,
  - enregistre toutes les métriques dans `self.history`,
  - affiche un résumé lisible pour l'epoch,
  - appelle l'early stopping avec la `valid_mse` :
    - si l'early stopping se déclenche, on sort de la boucle.
- renvoie l'historique de toutes les epochs (pratique pour tracer les courbes ensuite).


In [ ]:
class LSTMTrainer:
    def __init__(self, model, lr=1e-3, device="mps"):
        self.device = torch.device(device if torch.backends.mps.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.model = model.to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

        # 🔹 même structure d’historique que le GRU optimisé
        self.history = {
            "train_mse": [],
            "valid_mse": [],
            "train_rmse": [],
            "valid_rmse": [],
            "epoch_time": []
        }

        # 🔹 scheduler comme sur le GRU
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode="min",
            factor=0.5,
            patience=20
        )

    def train_epoch(self, loader):
        self.model.train()   #met le modèle en mode entrainemennt (utile pour dropout du LSTM voir cellule LSTMModel)
        losses = []

        for X, y in loader:
            X, y = X.to(self.device), y.to(self.device)

            pred = self.model(X)
            loss = self.loss_fn(pred, y)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            losses.append(loss.item())

        train_mse = np.mean(losses)
        train_rmse = np.sqrt(train_mse)
        return train_mse, train_rmse

    def eval_epoch(self, loader):
        self.model.eval()
        losses = []

        with torch.no_grad():
            for X, y in loader:
                X, y = X.to(self.device), y.to(self.device)
                pred = self.model(X)
                loss = self.loss_fn(pred, y)

                losses.append(loss.item())

        valid_mse = np.mean(losses)
        valid_rmse = np.sqrt(valid_mse)
        return valid_mse, valid_rmse

    def fit(self, train_loader, valid_loader, epochs=100, patience=20):
        es = EarlyStopping(patience=patience)

        for epoch in range(epochs):
            start_time = time.time()  # ⏱ début epoch

            train_mse, train_rmse = self.train_epoch(train_loader)
            valid_mse, valid_rmse = self.eval_epoch(valid_loader)

            epoch_time = time.time() - start_time

            self.history["train_mse"].append(train_mse)
            self.history["valid_mse"].append(valid_mse)
            self.history["train_rmse"].append(train_rmse)
            self.history["valid_rmse"].append(valid_rmse)
            self.history["epoch_time"].append(epoch_time)

            self.scheduler.step(valid_mse)

            print(
                f"[EPOCH {epoch+1:03d}] "
                f"train_mse={train_mse:.6f} | "
                f"valid_mse={valid_mse:.6f} | "
                f"valid_rmse={valid_rmse:.6f} | "
                f"temps_calcul={epoch_time:.3f}s"
            )

            es(valid_mse)
            if es.early_stop:
                print("Early stopping triggered.")
                break

        return self.history




## 6. Gestionnaire d'expériences : `ExperimentRunner`

Cette classe sert à **organiser plusieurs expériences** d'entraînement :

- `self.results` : liste qui contiendra un dictionnaire par expérience avec :
  - un nom lisible (`"name"`),
  - l'historique renvoyé par le trainer (`"history"`).

Méthode `run(self, name, model, trainer, train_loader, valid_loader, epochs, patience)` :

- Affiche un message pour indiquer le démarrage d'une expérimentation donnée (`name`).
- Appelle `trainer.fit(...)` avec les loaders et les hyperparamètres d'epochs/patience.
- Stocke dans `self.results` un dictionnaire `{"name": name, "history": history}`.
- Renvoie aussi `history` pour utilisation immédiate si besoin.

Méthode `plot(self)` :

- Crée une figure matplotlib.
- Pour chaque expérience dans `self.results` :
  - récupère la liste des `valid_rmse` dans `res["history"]["valid_rmse"]`,
  - trace la courbe de RMSE de validation avec une légende correspondant à l'expérience.
- Ajoute titre, labels, grille, légende pour comparer les différentes configurations
  (par ex. différentes tailles de hidden, différents learning rates, etc.).


In [ ]:
class ExperimentRunner:
    def __init__(self):
        self.results = []

    def run(self, name, model, trainer, train_loader, valid_loader, epochs=120, patience=20):
        print(f"\n Running experiment: {name}")
        history = trainer.fit(train_loader, valid_loader, epochs, patience)
        self.results.append({"name": name, "history": history})
        return history

    def plot(self):
        plt.figure(figsize=(14,6))

        for res in self.results:
            plt.plot(res["history"]["valid_rmse"], label=res["name"])

        plt.title("Comparaison des RMSE Validation (LSTM)")
        plt.xlabel("Epoch")
        plt.ylabel("RMSE")
        plt.legend()
        plt.grid(True)
        plt.show()


## 7. Préparation des données : lecture du CSV, features, cibles et DataLoader

Cette partie du code :

1. **Lit un fichier CSV** contenant des informations sur l'orbite du satellite et les erreurs :  
   - `df = pd.read_csv(csv_path, sep=";")` (par exemple).

2. **Nettoie certaines colonnes** :  
   - supprime d'éventuelles colonnes de correction déjà présentes (`dx_km`, `dy_km`, `dz_km`) pour éviter de les réutiliser comme entrées.

3. Définit les **colonnes d'entrée `X_cols`** :  
   - dates (`time_utc`, `tle_epoch`) converties en timestamps numériques,  
   - paramètres orbitaux (par ex. `mean_motion`, `orbital_speed_km_s`, dérivées, etc.),  
   - positions SGP4 (`x_sgp4_km`, `y_sgp4_km`, `z_sgp4_km`).

4. Définit les **colonnes de sortie `Y_cols`** (cibles) :  
   - souvent des erreurs de position par rapport à une référence plus précise (Horizons) :
     - par ex. `dx_horizons_sgp4_km`, `dy_...`, `dz_...`.

5. Convertit les dates `time_utc` et `tle_epoch` en **timestamps en secondes** depuis l'époque Unix :
   - `pd.to_datetime(...).astype("int64") / 1e9`.

6. Affiche les colonnes disponibles pour vérifier la bonne correspondance entre le CSV et le code :
   - `print("Colonnes du CSV :", list(df.columns))`.

7. Construit les matrices `X` et `Y` :
   - `X = df[X_cols].values.astype(np.float32)` : features en float32.
   - `Y = df[Y_cols].values.astype(np.float32)` : cibles en float32.

8. Crée le `SatelliteSequenceDataset` avec la longueur de séquence `seq_len` :
   - `dataset = SatelliteSequenceDataset(X, Y, seq_len=seq_len)`.

9. Configure le dataset en mode train/validation puis crée les DataLoader :
   - `train_set.set_mode("train")`, `valid_set.set_mode("valid")`.  
   - `train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)`  
   - `valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False)`.

L'objectif est de transformer le CSV brut en un flux de mini-batchs de séquences temporelles
prêts à être consommés par le modèle LSTM.


In [ ]:
if __name__ == "__main__":

    # ---------------------
    # Load real dataset
    # ---------------------
    csv_path = "datasetISS_200TLE.csv"  # adapter le chemin si besoin
    df = pd.read_csv(csv_path, sep=";")

    # Supprimer dx, dy, dz si présents
    for col in ["dx_km", "dy_km", "dz_km"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    # Colonnes d'entrée X
    X_cols = [
        "time_utc",
        "tle_epoch",
        "dt_since_tle_s",
        "mean_motion",
        "orbital_speed_km_s",
        "mean_motion_derivative",
        "altitude_drift_km_per_day",
        "bstar",
        "inclination_deg",
        "raan_deg",
        "eccentricity",
        "arg_perigee_deg",
        "mean_anomaly_deg",
        "rev_number",
        "x_sgp4_km",
        "y_sgp4_km",
        "z_sgp4_km",
    ]

    # Conversion des dates en timestamps numériques
    if "time_utc" in df.columns:
        df["time_utc"] = pd.to_datetime(df["time_utc"]).astype("int64") / 1e9  # secondes
    if "tle_epoch" in df.columns:
        df["tle_epoch"] = pd.to_datetime(df["tle_epoch"]).astype("int64") / 1e9

    # Debug: afficher les colonnes disponibles pour vérifier les noms réels
    print("Colonnes du CSV :", list(df.columns))

    # Petite fonction utilitaire pour retrouver une colonne Horizons même si le nom varie un peu
    def find_col(candidates):
        cols_lower = {c.lower().strip(): c for c in df.columns}
        for cand in candidates:
            key = cand.lower().strip()
            if key in cols_lower:
                return cols_lower[key]
        # Ultime recours : chercher en 'contains'
        for key, original in cols_lower.items():
            for cand in candidates:
                if cand.lower().strip() in key:
                    return original
        raise KeyError(f"Aucune colonne trouvée parmi {candidates} dans {df.columns}")

    # On essaie plusieurs variantes possibles des noms de colonnes Horizons
    x_h_col = find_col(["x_horizons_km", "x_horizon_km", "x_horizons"])
    y_h_col = find_col(["y_horizons_km", "y_horizon_km", "y_horizons"])
    z_h_col = find_col(["z_horizons_km", "z_horizon_km", "z_horizons"])

    # Calcul des erreurs SGP4 -> Horizons
    df["err_x"] = df[x_h_col] - df["x_sgp4_km"]
    df["err_y"] = df[y_h_col] - df["y_sgp4_km"]
    df["err_z"] = df[z_h_col] - df["z_sgp4_km"]

    # Matrices numpy
    X = df[X_cols].values.astype(np.float32)
    Y = df[["err_x", "err_y", "err_z"]].values.astype(np.float32)

    # Normalisation simple (z-score) de X
    X_mean = X.mean(axis=0, keepdims=True)
    X_std = X.std(axis=0, keepdims=True) + 1e-8
    X = (X - X_mean) / X_std

    seq_len = 128
    batch_size = 32

    dataset = SatelliteSequenceDataset(X, Y, seq_len=seq_len)

    train_set, valid_set = dataset, dataset  # same object, mode changes
    train_set.set_mode("train")
    valid_set.set_mode("valid")

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False)

 
    

## 8. Lancement des expériences et comparaison des erreurs

Cette dernière partie du script :

1. Crée une instance de `ExperimentRunner` :
   - `runner = ExperimentRunner()`.

2. Déclare une ou plusieurs configurations de modèles LSTM :
   - Par exemple :
     - `model1 = LSTMModel(input_size=17, hidden_size=64, num_layers=1)`  
       → un LSTM avec 64 neurones cachés, 1 couche.  
     - On associe un `LSTMTrainer` avec un certain `lr` (par ex. `1e-4`).

3. Lance chaque expérience avec `runner.run(...)` :
   - En lui passant :
     - un nom lisible (`"LSTM_64_hidden_lr1e-4"`),
     - le modèle,
     - le trainer,
     - les `train_loader` et `valid_loader`,
     - le nombre d'epochs et la patience.

4. Après les expériences, appelle `runner.plot()` :
   - Trace les courbes de `valid_rmse` de chaque expérience sur la même figure.

5. En fin de script (partie spécialisée SGP4 / Horizons) :
   - Récupère :
     - la position SGP4 (`sgp4_pos`),
     - la position de référence Horizons (`horizons_pos`),
     - la correction prédite par le modèle (`pred_err`).

   - Calcule la **position corrigée** :
     - `model_pos = sgp4_pos + pred_err`.

   - Calcule les **erreurs 3D** :
     - `err_sgp4 = ||horizons_pos - sgp4_pos||` (norme 3D).  
     - `err_model = ||horizons_pos - model_pos||`.

   - Trace deux courbes :
     - erreur SGP4 → Horizons,
     - erreur modèle (SGP4 + correction LSTM) → Horizons,
     pour montrer l'amélioration apportée par le modèle.

Cette partie permet de *visualiser concrètement* l'apport du réseau LSTM par rapport
au modèle SGP4 brut, en termes d'erreur 3D sur la position du satellite.


In [ ]:
    # ---------------------
    # RUN EXPERIMENTS
    # ---------------------

runner = ExperimentRunner()

    # Experiment 1
    model1 = LSTMModel(input_size=17, hidden_size=64, num_layers=1)
    trainer1 = LSTMTrainer(model1, lr=1e-4)

    runner.run("LSTM_64_hidden_lr1e-4", model1, trainer1, train_loader, valid_loader)

    '''# Experiment 2
    model2 = LSTMModel(input_size=18, hidden_size=128, num_layers=2)
    trainer2 = LSTMTrainer(model2, lr=5e-4)

    runner.run("LSTM_128_hidden_lr5e-4", model2, trainer2, train_loader, valid_loader)
    '''
    # ---------------------
    # Plot results (RMSE)
    # ---------------------
    runner.plot()

    # ---------------------
    # Comparaison SGP4 vs Horizons vs modèle (sur la validation) — comme pour le GRU
    # ---------------------
    model1.eval()
    device = trainer1.device

    valid_set.set_mode("valid")
    split = dataset.split
    n_valid = dataset.X_valid.shape[0]

    all_idx = []
    pred_err_list = []

    with torch.no_grad():
        for i in range(n_valid - seq_len):
            X_seq, _ = valid_set[i]
            X_seq = X_seq.unsqueeze(0).to(device)
            pred_err = model1(X_seq).cpu().numpy()[0]   # (3,)
            pred_err_list.append(pred_err)

            idx_global = split + i + seq_len - 1
            all_idx.append(idx_global)

    all_idx = np.array(all_idx)
    pred_err = np.array(pred_err_list)

    sgp4_pos = df.loc[all_idx, ["x_sgp4_km", "y_sgp4_km", "z_sgp4_km"]].values
    horizons_pos = df.loc[all_idx, [x_h_col, y_h_col, z_h_col]].values

    model_pos = sgp4_pos + pred_err

    err_sgp4 = np.linalg.norm(horizons_pos - sgp4_pos, axis=1)
    err_model = np.linalg.norm(horizons_pos - model_pos, axis=1)

    plt.figure(figsize=(12, 5))
    plt.plot(err_sgp4, label="Erreur 3D SGP4 → Horizons")
    plt.plot(err_model, label="Erreur 3D Modèle LSTM (SGP4 + correction)", alpha=0.8)
    plt.xlabel("Indice dans la série de validation")
    plt.ylabel("Erreur 3D (km)")
    plt.title("Comparaison des erreurs 3D : SGP4 vs modèle LSTM corrigé")
    plt.legend()
    plt.grid(True)
    plt.show()